# Download CMEMS data

Download and cache all CMEMS datasets needed by the Baltic drifter notebooks.

## Imports

In [ ]:
from pathlib import Path

import copernicusmarine

## Parameters

In [ ]:
TIME_START = "2023-04-24"
TIME_END = "2023-05-10"
LON_MIN = 9.0
LON_MAX = 13.0
LAT_MIN = 53.5
LAT_MAX = 56.0
OUTPUT_DIR = "data/cmems"

## Setup

In [ ]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

## Physics (currents)

In [ ]:
ds_phy = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_bal_phy_anfc_PT1H-i",
    service="arco-geo-series",
).sel(
    longitude=slice(LON_MIN, LON_MAX),
    latitude=slice(LAT_MIN, LAT_MAX),
    time=slice(TIME_START, TIME_END),
    depth=slice(0, 5),
)[["uo", "vo"]].load()

out_path = output_dir / "cmems_mod_bal_phy_anfc_PT1H-i.nc"
ds_phy.to_netcdf(out_path)
print(f"Physics: {dict(ds_phy.dims)}")
print(f"  {out_path.name}: {out_path.stat().st_size / 1e6:.1f} MB")

## Waves

In [ ]:
WAVE_VARS = [
    "VSDX", "VSDY",          # surface Stokes drift components
    "VHM0", "VTPK",          # total Hs and peak period
    "VHM0_WW", "VTM01_WW", "VMDR_WW",     # wind wave partition
    "VHM0_SW1", "VTM01_SW1", "VMDR_SW1",  # swell partition 1
    "VHM0_SW2", "VTM01_SW2", "VMDR_SW2",  # swell partition 2
]

ds_wav = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_bal_wav_anfc_PT1H-i",
    service="arco-geo-series",
).sel(
    longitude=slice(LON_MIN, LON_MAX),
    latitude=slice(LAT_MIN, LAT_MAX),
    time=slice(TIME_START, TIME_END),
)[WAVE_VARS].load()

out_path = output_dir / "cmems_mod_bal_wav_anfc_PT1H-i.nc"
ds_wav.to_netcdf(out_path)
print(f"Waves: {dict(ds_wav.dims)}")
print(f"  {out_path.name}: {out_path.stat().st_size / 1e6:.1f} MB")

## Static (land mask)

In [ ]:
ds_static = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_bal_phy_anfc_static",
    service="static-arco",
)[["mask"]].load()

out_path = output_dir / "cmems_mod_bal_phy_anfc_static.nc"
ds_static.to_netcdf(out_path)
print(f"Static: {dict(ds_static.dims)}")

## Summary

In [ ]:
for f in sorted(output_dir.glob("*.nc")):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name}: {size_mb:.1f} MB")